In [ ]:
from importlib import reload
import torch
from torch.nn import functional as F
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import numpy as np
#Import graph based
from graph_based.utils.loader import  get_dataset
from graph_based.models.gnn.layers import GraphConvolution
from graph_based.models import resnet
from graph_based.models.hyper_models import HybridGNN
from graph_based.train import train_hybrid
from graph_based.utils.evaluate import eval_normal, eval_hybrid
#Import SSL
from SSL_method import ssl
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import matplotlib.patches as mpatches

In [ ]:
def get_data_graph(nodes,links):
    dataset, _ , adj = get_dataset(nodes, links, nodes.shape[0]) 
    return dataset, adj

In [ ]:
features = pd.read_csv("./graph_based/dataset/node_2label_balance.csv").to_numpy()
links = pd.read_csv("./graph_based/dataset/edge_2label_balance.csv").to_numpy().astype(float)
links_df = pd.read_csv("./graph_based/dataset/edge_2label_balance.csv")
train_xgb = pd.read_csv("./graph_based/dataset/train.csv").to_numpy()

In [ ]:
X_pool, X_gnn_test, X_xgb_pool, X_xgb_test, y_pool, y_test = train_test_split(features, train_xgb, features[:,-1],test_size=0.2, random_state= 42)
X_gnn_train, X_gnn_unlabeled, X_xgb_train, X_xgb_unlabeled, y_train, y_unlabel = train_test_split(X_pool, X_xgb_pool, y_pool, test_size=0.5, random_state= 42)


In [ ]:
# dataset_train, nrm_adj_train, adjs_train = get_dataset(X_gnn_train, links, X_gnn_train.shape[0]) 
# dataset_test, nrm_adj_test, adjs_test = get_dataset(X_gnn_test, links, X_gnn_test.shape[0]) 
dataset_train, adjs_train = get_data_graph(X_gnn_train, links)
dataset_val, adj_val = get_data_graph(X_gnn_test, links)


In [ ]:
nets = [resnet.ResNet(GraphConvolution,
             in_features=features.shape[1] - 2,
             cfg=[16, 16, 32, 16],
             n_residual_units=[2, 2, 2],
             hidden_layers_fc= [16,8],
             num_classes=None) for _ in range(features.shape[1] - 2)]
model_graph = HybridGNN(2, 'cuda', *nets).cuda()

In [ ]:
model_graph.load_state_dict(torch.load("mode_new.pt", map_location='cpu'))
pred, f1_resnet_test, acc_resnet_test, prob = eval_hybrid(model_graph, dataset_val.data, dataset_val.targets, adj_val)
print(acc_resnet_test)

In [ ]:
XGB1 = xgb.XGBClassifier(eval_metric = "logloss",objective = "binary:logistic", booster = "gbtree", tree_method = "gpu_hist",
                             max_depth = 4, gamma = 2, learning_rate = 0.005, n_estimators = 300, subsample = 0.7 , colsample_bytree = 0.7, min_child_weight = 4, use_label_encoder  = False )
evalset1 = [(X_xgb_train[:,1:-1], y_train), (X_xgb_test[:,1:-1], y_test)]
XGB1 = XGB1.fit(X_xgb_train[:,1:-1], y_train , eval_metric='logloss', eval_set=evalset1)
prediction_test1 = XGB1.predict(X_xgb_test[:,1:-1])
f1_xgb = f1_score(y_test,prediction_test1)
acc_xgb = accuracy_score(y_test,prediction_test1)

In [ ]:
print(f1_resnet_test, acc_resnet_test)
print(f1_xgb, acc_xgb)

In [ ]:
reload(ssl)

In [ ]:
model_graph.load_state_dict(torch.load("mode_new.pt", map_location='cpu'))
SSL = ssl.SSLClassifier(estimators= [XGB1,model_graph],
                    X_trains= [X_xgb_train, X_gnn_train],
                    y_train= y_train,
                    X_vals= [X_xgb_test, X_gnn_test],
                    y_val= y_test,
                    links=links,
                    p=50,
                    n=50,
                    k=30,
                    unlabeled_pool_size=500,
                    type_ssl='al',
                    type_estimator='gb'
                    )
acc_al1,f1_al1,acc_al2,f1_al2 = SSL.fit(X_pools = [X_xgb_unlabeled, X_gnn_unlabeled],
                y_pool = y_pool)

In [ ]:
hnd2, = plt.plot(acc_al2,'c',label='ActiveLearning')

In [ ]:
output = pd.read_csv("output.csv")
output['acc_al2'] = acc_al2
output.to_csv("output.csv")

In [ ]:
output

In [ ]:
output.to_excel('output.xlsx')